In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# World Cup 2026 simulation with LightGBM

Train a match-result model on the gold feature table, then simulate the World Cup with probabilistic match sampling.

In [10]:
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

from world_cup_features_fifa import (
    get_training_feature_columns,
    build_team_feature_store,
    prepare_match_row,
    prepare_world_cup_matches,
)
from world_cup_simulation import simulate_world_cup

In [11]:
gold_path = '/Users/ormenessevinicius/Library/CloudStorage/OneDrive-TheBostonConsultingGroup,Inc/Documents/GitHub/analise_times/football_analysis/data/gold_fifa_partidas/part-0.parquet'
fixtures_path = "world_cup_2026_games.csv"

df = pd.read_parquet(gold_path)
fixtures = pd.read_csv(fixtures_path)

In [12]:
df.shape*2

(59970, 329, 59970, 329)

In [13]:
29985*2

59970

In [14]:
df.head()

,match_id,match_date,snapshot_date,competition_code,competition_group,match_type,season,home_team,away_team,home_goals,...,away_xi_overall_mean_MID,away_xi_overall_mean_FWD,away_squad_source,away_team_avg_score,away_team_attack_score,away_team_mid_score,away_team_def_score,away_team_gk_score,home_win_rate_last10,away_win_rate_last10
0,INT-Friendlies|2015|2015-01-04|Bahrain|Jordan,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Bahrain,Jordan,1,...,61.5,NaN,nationality_top,59.818182,NaN,58.500,62.00,NaN,0.0,0.0
1,INT-Friendlies|2015|2015-01-04|Bahrain|Jordan:...,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Jordan,Bahrain,0,...,NaN,NaN,nationality_top,68.000000,NaN,NaN,74.00,NaN,0.0,0.0
2,INT-Friendlies|2015|2015-01-04|Iran|Iraq,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Iran,Iraq,1,...,63.0,NaN,nationality_top,62.571429,NaN,63.125,63.75,62.0,0.0,0.0
3,INT-Friendlies|2015|2015-01-04|Iran|Iraq:mirror,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Iraq,Iran,0,...,75.0,63.0,nationality_top,66.500000,67.333333,71.000,66.50,67.0,0.0,0.0
4,INT-Friendlies|2015|2015-01-04|South Africa|Za...,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,South Africa,Zambia,1,...,68.0,71.0,nationality_top,69.250000,77.666667,65.750,74.50,NaN,0.0,0.0


In [15]:
feature_columns = get_training_feature_columns(df)
X = df[feature_columns].fillna(0).astype("float32")
y = df["target"]

print(X.shape)
print(y.value_counts())

(59970, 252)
target
W    21709
L    21709
D    16552
Name: count, dtype: int64


In [16]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

def objective(trial):
    params = {
        "objective": "multiclass",
        "num_class": len(le.classes_),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": 1,
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y_encoded, cv=cv, scoring="accuracy", n_jobs=1)
    return scores.mean()

import os
N_TRIALS = int(os.environ.get("WC_OPTUNA_TRIALS", "30"))  # default 30; reduza p/ rodar rápido
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)

print("Best score:", study.best_value)
print("Best params:", study.best_params)


[I 2026-06-14 08:35:32,292] A new study created in memory with name: no-name-53b0afdf-ab14-4f93-831e-a578b9d8495a
[I 2026-06-14 08:47:14,776] Trial 0 finished with value: 0.44975821243955316 and parameters: {'n_estimators': 190, 'learning_rate': 0.010401313924185954, 'num_leaves': 60, 'max_depth': 3, 'min_child_samples': 52, 'subsample': 0.8716817016322007, 'colsample_bytree': 0.7646453720084193, 'reg_alpha': 0.08090522586920285, 'reg_lambda': 0.1690044613906582}. Best is trial 0 with value: 0.44975821243955316.
[I 2026-06-14 09:03:23,261] Trial 1 finished with value: 0.45210938802734696 and parameters: {'n_estimators': 39, 'learning_rate': 0.020487900454356545, 'num_leaves': 54, 'max_depth': 9, 'min_child_samples': 77, 'subsample': 0.8182372914779706, 'colsample_bytree': 0.9410516461643263, 'reg_alpha': 0.0015064100384718177, 'reg_lambda': 3.616333661129018}. Best is trial 1 with value: 0.45210938802734696.
[I 2026-06-14 09:21:41,615] Trial 2 finished with value: 0.45204268801067204 a

Best score: 0.45389361347340335
Best params: {'n_estimators': 442, 'learning_rate': 0.01427516665786675, 'num_leaves': 16, 'max_depth': 6, 'min_child_samples': 20, 'subsample': 0.9228081683650677, 'colsample_bytree': 0.9310793089345103, 'reg_alpha': 0.05667501215601239, 'reg_lambda': 0.8998882018812595}


In [17]:
best_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    metric="multi_logloss",
    boosting_type="gbdt",
    verbosity=-1,
    random_state=42,
    n_jobs=1,
    **study.best_params,
)

best_model.fit(X, y_encoded)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df.head(20)

,feature,importance
250,home_win_rate_last10,554
251,away_win_rate_last10,553
231,away_xi_value_eur_mean,333
210,home_xi_value_eur_mean,330
208,home_xi_overall_mean,280
229,away_xi_overall_mean,270
190,away_p10_value_eur,237
86,home_p10_value_eur,226
233,away_xi_pace_mean,217
232,away_xi_age_mean,202


In [ ]:
import json

best_model._Booster.save_model('fifa_best_model.lgb')

# Meta do modelo de RESULTADO lido pelo app (streamlit_app.py): feature_columns
# (já com FORMAÇÃO + POSIÇÕES + stats médios por grupo) e as classes na ordem do
# LabelEncoder (= ordem das colunas do predict). Reescrever aqui mantém o app em
# sincronia com as features novas após re-treino.
with open("fifa_best_model_meta.json", "w", encoding="utf-8") as fh:
    json.dump({"feature_columns": feature_columns,
               "classes": list(le.classes_)}, fh, ensure_ascii=False, indent=2)
print(f"saved fifa_best_model.lgb + meta ({len(feature_columns)} features)")

In [19]:
store = build_team_feature_store(df, feature_columns)

wc_feature_rows = prepare_world_cup_matches(fixtures, store)
wc_feature_rows[["MatchNumber", "HomeTeam", "AwayTeam", "home_team_has_history", "away_team_has_history"]].head(10)

,MatchNumber,HomeTeam,AwayTeam,home_team_has_history,away_team_has_history
0,1,Mexico,South Africa,True,True
1,2,Korea Republic,Czechia,True,True
2,3,Canada,Bosnia and Herzegovina,True,True
3,4,USA,Paraguay,True,True
4,5,Haiti,Scotland,True,True
5,6,Australia,Türkiye,True,True
6,7,Brazil,Morocco,True,True
7,8,Qatar,Switzerland,True,True
8,9,Côte d'Ivoire,Ecuador,True,True
9,10,Germany,Curaçao,True,True


## Goal-score models (home & away)

Two extra LightGBM classifiers predicting goals per side — classes **0,1,2,3,4,5+** (goals ≥5 grouped). Same features as the result model; `predict_proba` gives the probability of each scoreline count.


In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

GOAL_CAP = 5  # goals >= 5 -> class "5+"

def tune_goal_model(target_col):
    """LightGBM tunado por Optuna para a contagem de gols (0..5+).

    Otimiza o neg log-loss em CV 5-fold (calibração de probabilidade — métrica
    adequada para distribuições de placar), espelhando o uso de Optuna do modelo
    de resultado. Usa o mesmo N_TRIALS (definido na célula do modelo de resultado).
    """
    y_goals = df[target_col].clip(upper=GOAL_CAP).astype(int)
    classes = sorted(y_goals.unique().tolist())
    n_classes = len(classes)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def objective(trial):
        params = {
            "objective": "multiclass", "num_class": n_classes, "metric": "multi_logloss",
            "boosting_type": "gbdt", "verbosity": -1, "random_state": 42, "n_jobs": 1,
            "n_estimators": trial.suggest_int("n_estimators", 10, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
        model = lgb.LGBMClassifier(**params)
        scores = cross_val_score(model, X, y_goals, cv=cv, scoring="neg_log_loss", n_jobs=1)
        return scores.mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
    best = lgb.LGBMClassifier(
        objective="multiclass", num_class=n_classes, metric="multi_logloss",
        boosting_type="gbdt", verbosity=-1, random_state=42, n_jobs=1, **study.best_params,
    )
    best.fit(X, y_goals)
    print(f"{target_col}: best neg_log_loss={study.best_value:.4f}")
    return best, classes

home_goals_model, home_goal_classes = tune_goal_model("home_goals")
away_goals_model, away_goal_classes = tune_goal_model("away_goals")
goal_labels = [str(c) if c < GOAL_CAP else f"{GOAL_CAP}+" for c in home_goal_classes]
goal_labels


[I 2026-06-14 10:07:38,517] A new study created in memory with name: no-name-3a81247b-6f31-4c5e-badc-8f357ab3e3f5
[I 2026-06-14 10:08:23,297] Trial 0 finished with value: -1.4389734041467293 and parameters: {'n_estimators': 46, 'learning_rate': 0.012611255417317719, 'num_leaves': 87, 'max_depth': 7, 'min_child_samples': 52, 'subsample': 0.769466195684515, 'colsample_bytree': 0.772329786195672, 'reg_alpha': 0.3217202918648366, 'reg_lambda': 0.2244682819558865}. Best is trial 0 with value: -1.4389734041467293.
[I 2026-06-14 10:13:47,904] Trial 1 finished with value: -1.5139002711175222 and parameters: {'n_estimators': 331, 'learning_rate': 0.06549905408217999, 'num_leaves': 103, 'max_depth': 8, 'min_child_samples': 40, 'subsample': 0.8042358801703513, 'colsample_bytree': 0.9569185889982715, 'reg_alpha': 0.0019187674472919264, 'reg_lambda': 5.45093850074286}. Best is trial 0 with value: -1.4389734041467293.
[I 2026-06-14 10:16:24,590] Trial 2 finished with value: -1.4283545173188075 and p

home_goals: best neg_log_loss=-1.4277


[I 2026-06-14 11:51:21,962] Trial 0 finished with value: -1.4357103793605241 and parameters: {'n_estimators': 218, 'learning_rate': 0.06901191543394651, 'num_leaves': 70, 'max_depth': 4, 'min_child_samples': 71, 'subsample': 0.9715023407976913, 'colsample_bytree': 0.8968017283700335, 'reg_alpha': 0.7797556902018173, 'reg_lambda': 0.0027249447794366133}. Best is trial 0 with value: -1.4357103793605241.
[I 2026-06-14 11:54:15,144] Trial 1 finished with value: -1.4320117949209483 and parameters: {'n_estimators': 172, 'learning_rate': 0.010290839513352383, 'num_leaves': 82, 'max_depth': 8, 'min_child_samples': 46, 'subsample': 0.8189448352708443, 'colsample_bytree': 0.8943621848582405, 'reg_alpha': 0.09135421801127473, 'reg_lambda': 0.052090485976129086}. Best is trial 1 with value: -1.4320117949209483.
[I 2026-06-14 11:58:19,589] Trial 2 finished with value: -1.4730873924763679 and parameters: {'n_estimators': 328, 'learning_rate': 0.050625064587435126, 'num_leaves': 48, 'max_depth': 7, '

away_goals: best neg_log_loss=-1.4282


['0', '1', '2', '3', '4', '5+']

In [21]:
import json

home_goals_model._Booster.save_model("fifa_home_goals_model.lgb")
away_goals_model._Booster.save_model("fifa_away_goals_model.lgb")

for side, classes in [("home", home_goal_classes), ("away", away_goal_classes)]:
    labels = [str(c) if c < GOAL_CAP else f"{GOAL_CAP}+" for c in classes]
    with open(f"fifa_{side}_goals_model_meta.json", "w", encoding="utf-8") as fh:
        json.dump({"feature_columns": feature_columns, "classes": classes, "labels": labels,
                   "target": f"{side}_goals", "goal_cap": GOAL_CAP}, fh, ensure_ascii=False, indent=2)
print("saved fifa_home_goals_model.lgb / fifa_away_goals_model.lgb")


saved fifa_home_goals_model.lgb / fifa_away_goals_model.lgb


In [22]:
from world_cup_simulation import predict_goal_distributions

gd = predict_goal_distributions("Brazil", "Germany", store,
                                home_goals_model, away_goals_model, goal_labels, goal_labels)
print("most likely score:", gd["home_goals"], "-", gd["away_goals"])
pd.DataFrame({"Brazil (home)": gd["home_dist"], "Germany (away)": gd["away_dist"]})


most likely score: 1 - 1


/opt/anaconda3/envs/general_12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/general_12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Brazil (home),Germany (away)
0,0.281197,0.168521
1,0.390749,0.294924
2,0.205456,0.279805
3,0.085622,0.151347
4,0.026532,0.094641
5+,0.010444,0.010762


In [23]:
example_match = prepare_match_row("Brazil", "Germany", store).fillna(0).astype("float32")
example_proba = pd.DataFrame(
    [best_model.predict_proba(example_match)[0]],
    columns=[f"proba_{c}" for c in le.classes_],
)
example_proba

,proba_D,proba_L,proba_W
0,0.252334,0.50468,0.242986


In [24]:
simulation = simulate_world_cup(
    fixtures_df=fixtures,
    model=best_model,
    store=store,
    label_encoder=le,
    rng_seed=42,
)

simulation["knockout_matches"]

,MatchNumber,RoundNumber,Stage,HomeTeam,AwayTeam,result,winner,most_likely_result,most_likely_winner,proba_W,proba_D,proba_L,home_goals,away_goals
0,73,4,Round of 32,Mexico,Bosnia and Herzegovina,D,Mexico,D,Mexico,0.373538,0.379576,0.246886,None,None
1,74,4,Round of 32,Germany,Scotland,W,Germany,W,Germany,0.664666,0.182329,0.153005,None,None
2,75,4,Round of 32,Netherlands,Brazil,W,Netherlands,W,Netherlands,0.513979,0.241262,0.244759,None,None
3,76,4,Round of 32,Morocco,Japan,L,Japan,L,Japan,0.336506,0.300764,0.362730,None,None
4,77,4,Round of 32,France,IR Iran,W,France,W,France,0.793476,0.122456,0.084067,None,None
5,78,4,Round of 32,Côte d'Ivoire,Senegal,W,Côte d'Ivoire,W,Côte d'Ivoire,0.390035,0.270226,0.339739,None,None
6,79,4,Round of 32,Czechia,Saudi Arabia,W,Czechia,W,Czechia,0.577447,0.272305,0.150247,None,None
7,80,4,Round of 32,England,Ecuador,W,England,W,England,0.706211,0.176488,0.117302,None,None
8,81,4,Round of 32,Türkiye,Canada,W,Türkiye,W,Türkiye,0.528753,0.278030,0.193217,None,None
9,82,4,Round of 32,Belgium,Korea Republic,W,Belgium,W,Belgium,0.558578,0.283411,0.158011,None,None


In [25]:
simulation['standings']

,team,pts,gf,ga,gd,wins,draws,losses,Group,rank
0,Czechia,7,2,0,2,2,1,0,Group A,1
1,Mexico,5,1,0,1,1,2,0,Group A,2
2,Korea Republic,4,1,1,0,1,1,1,Group A,3
3,South Africa,0,0,3,-3,0,0,3,Group A,4
4,Switzerland,9,3,0,3,3,0,0,Group B,1
5,Bosnia and Herzegovina,6,2,1,1,2,0,1,Group B,2
6,Canada,3,1,2,-1,1,0,2,Group B,3
7,Qatar,0,0,3,-3,0,0,3,Group B,4
8,Morocco,9,3,0,3,3,0,0,Group C,1
9,Brazil,6,2,1,1,2,0,1,Group C,2


In [26]:
simulation['group_matches']

,MatchNumber,RoundNumber,Stage,HomeTeam,AwayTeam,result,winner,most_likely_result,most_likely_winner,proba_W,proba_D,proba_L,home_goals,away_goals
0,1,1,Group A,Mexico,South Africa,W,Mexico,W,Mexico,0.534327,0.291110,0.174563,None,None
1,2,1,Group A,Korea Republic,Czechia,L,Czechia,L,Czechia,0.292488,0.317592,0.389921,None,None
2,25,2,Group A,Czechia,South Africa,W,Czechia,W,Czechia,0.550345,0.282012,0.167643,None,None
3,28,2,Group A,Mexico,Korea Republic,D,Korea Republic,D,Korea Republic,0.296954,0.375753,0.327292,None,None
4,53,3,Group A,Czechia,Mexico,D,Czechia,D,Czechia,0.352749,0.366313,0.280938,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,22,1,Group L,England,Croatia,W,England,W,England,0.523689,0.230572,0.245738,None,None
68,45,2,Group L,England,Ghana,W,England,W,England,0.702279,0.173984,0.123736,None,None
69,46,2,Group L,Panama,Croatia,L,Croatia,L,Croatia,0.087690,0.155959,0.756351,None,None
70,67,3,Group L,Panama,England,L,England,L,England,0.065521,0.109079,0.825400,None,None


In [27]:
simulation["knockout_matches"]

,MatchNumber,RoundNumber,Stage,HomeTeam,AwayTeam,result,winner,most_likely_result,most_likely_winner,proba_W,proba_D,proba_L,home_goals,away_goals
0,73,4,Round of 32,Mexico,Bosnia and Herzegovina,D,Mexico,D,Mexico,0.373538,0.379576,0.246886,None,None
1,74,4,Round of 32,Germany,Scotland,W,Germany,W,Germany,0.664666,0.182329,0.153005,None,None
2,75,4,Round of 32,Netherlands,Brazil,W,Netherlands,W,Netherlands,0.513979,0.241262,0.244759,None,None
3,76,4,Round of 32,Morocco,Japan,L,Japan,L,Japan,0.336506,0.300764,0.362730,None,None
4,77,4,Round of 32,France,IR Iran,W,France,W,France,0.793476,0.122456,0.084067,None,None
5,78,4,Round of 32,Côte d'Ivoire,Senegal,W,Côte d'Ivoire,W,Côte d'Ivoire,0.390035,0.270226,0.339739,None,None
6,79,4,Round of 32,Czechia,Saudi Arabia,W,Czechia,W,Czechia,0.577447,0.272305,0.150247,None,None
7,80,4,Round of 32,England,Ecuador,W,England,W,England,0.706211,0.176488,0.117302,None,None
8,81,4,Round of 32,Türkiye,Canada,W,Türkiye,W,Türkiye,0.528753,0.278030,0.193217,None,None
9,82,4,Round of 32,Belgium,Korea Republic,W,Belgium,W,Belgium,0.558578,0.283411,0.158011,None,None
